# RetailIQ - Enterprise Risk Analysis & Assessment

This notebook provides a comprehensive, empirical evaluation of **Enterprise Retail Risks** using the cleaned Online Retail dataset:
- **Regional Risk**: Geographic revenue concentration across 37 global markets
- **Customer Risk**: Whale account revenue dependency and customer churn in the At-Risk cohort
- **Demand & Seasonality Risk**: Monthly revenue volatility and Q4 holiday surge vs Q1 slump
- **Inventory Risk**: Long-tail dead stock and capital holding costs
- **Product Risk**: Pareto skew and top-SKU stockout vulnerability
- **Sales Risk**: Single-order customer acquisition leakage
- **Operational Risk**: Volume surge fulfillment strain and cross-border logistics complexity
- **Risk Assessment Matrix**: 8 core risks mapped by Probability, Impact, Severity, and Mitigation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load cleaned retail dataset and segmentation results
df = pd.read_csv('../data/online_retail_clean.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
segments_df = pd.read_csv('../data/customer_segments.csv')

total_sales = float(df['Sales'].sum())
print(f'Total Transactions: {len(df):,}')
print(f'Total Revenue: ${total_sales:,.2f}')
print(f'Unique Customers: {df["Customer ID"].nunique():,}')
print(f'Unique Countries: {df["Country"].nunique():,}')

## 1. Regional Concentration Risk
Evaluating geographic exposure and country-level revenue distribution.

In [2]:
country_sales = df.groupby('Country')['Sales'].sum().sort_values(ascending=False)
uk_sales = country_sales.get('United Kingdom', 0)
uk_share = (uk_sales / total_sales) * 100

print(f'United Kingdom Revenue: ${uk_sales:,.2f} ({uk_share:.2f}%)')
print(f'Non-UK Revenue (36 Countries): ${total_sales - uk_sales:,.2f} ({100 - uk_share:.2f}%)')

# Top 5 Export Markets
top_exports = country_sales.drop('United Kingdom', errors='ignore').head(5)
for c, s in top_exports.items():
    print(f'  {c}: ${s:,.2f} ({(s / total_sales) * 100:.2f}%)')

## 2. Customer Concentration & Churn Risk
Analyzing dependency on whale accounts and dormant revenue exposure.

In [3]:
cust_sales = df.groupby('Customer ID')['Sales'].sum().sort_values(ascending=False)
top_10 = cust_sales.head(10).sum()
top_1pct_n = max(1, int(len(cust_sales) * 0.01))
top_1pct = cust_sales.head(top_1pct_n).sum()

print(f'Top 10 Accounts Revenue: ${top_10:,.2f} ({(top_10/total_sales)*100:.2f}%)')
print(f'Top 1% Accounts ({top_1pct_n}) Revenue: ${top_1pct:,.2f} ({(top_1pct/total_sales)*100:.2f}%)')

# At-Risk Cohort from RFM Segmentation
at_risk = segments_df[segments_df['Customer Segment'] == 'At-Risk Customers']
print(f'At-Risk Accounts: {len(at_risk):,} ({(len(at_risk)/len(segments_df))*100:.2f}% of customer base)')
print(f'At-Risk Historical Spend: ${at_risk["Monetary"].sum():,.2f}')
print(f'Average Inactivity: {at_risk["Recency"].mean():.1f} days')

## 3. Demand Volatility & Seasonality Risk
Measuring monthly coefficient of variation (CV) and holiday demand swing.

In [4]:
monthly = df.set_index('InvoiceDate').resample('MS')['Sales'].sum()
full_m = monthly.loc['2009-12-01':'2010-11-30']
m_mean = full_m.mean()
m_std = full_m.std()
m_cv = (m_std / m_mean) * 100
q1_trough = monthly.loc['2010-01-01':'2010-02-28'].mean()
q4_peak = monthly.loc['2010-10-01':'2010-11-30'].mean()
swing = ((q4_peak - q1_trough) / q1_trough) * 100

print(f'Monthly Mean: ${m_mean:,.2f}, Std Dev: ${m_std:,.2f}')
print(f'Demand Volatility (CV): {m_cv:.1f}%')
print(f'Q4 vs Q1 Demand Swing: +{swing:.1f}%')

## 4. Product & Inventory Risk
Evaluating Pareto skew, stockout sensitivity, and slow-moving dead stock.

In [5]:
prod_sales = df.groupby('Description')['Sales'].sum().sort_values(ascending=False)
total_skus = len(prod_sales)
pareto_80_n = int((prod_sales.cumsum() <= total_sales * 0.80).sum())
dead_stock_n = int((prod_sales < 100.0).sum())

print(f'Total Catalog SKUs: {total_skus:,}')
print(f'Top SKUs generating 80% revenue: {pareto_80_n:,} ({(pareto_80_n/total_skus)*100:.1f}%)')
print(f'Dead-Stock SKUs (<$100 sales): {dead_stock_n:,} ({(dead_stock_n/total_skus)*100:.1f}%)')

## 5. Risk Assessment Matrix
Summarizing the 8 core retail risks with likelihood, impact, and mitigation protocols.

In [6]:
import sys
sys.path.append('..')
from src.risk import RiskAnalyzer

analyzer = RiskAnalyzer(df, segments_df)
report = analyzer.analyze_all()
matrix_df = pd.DataFrame(report['matrix'])
matrix_df[['id', 'category', 'identified_risk', 'probability', 'impact', 'risk_level']]